In [4]:
import os, sys, platform, subprocess, time, multiprocessing
import psutil
import numpy as np

print("="*50)
print("CPU")
print("="*50)
print(f"Physical cores : {psutil.cpu_count(logical=False)}")
print(f"Logical cores  : {psutil.cpu_count(logical=True)}")
freq = psutil.cpu_freq()
if freq:
    print(f"Frequency      : {freq.current:.0f} / {freq.max:.0f} MHz")

print("\n" + "="*50)
print("RAM")
print("="*50)
mem = psutil.virtual_memory()
print(f"Total          : {mem.total/1e9:.1f} GB")
print(f"Available      : {mem.available/1e9:.1f} GB")
print(f"Used           : {mem.used/1e9:.1f} GB  ({mem.percent:.1f}%)")
swap = psutil.swap_memory()
print(f"Swap total     : {swap.total/1e9:.1f} GB")

print("\n" + "="*50)
print("DISK")
print("="*50)
for p in psutil.disk_partitions():
    try:
        u = psutil.disk_usage(p.mountpoint)
        print(f"{p.mountpoint:10s}  Total:{u.total/1e9:.0f}GB  Free:{u.free/1e9:.0f}GB")
    except:
        pass

print("\n" + "="*50)
print("GPU")
print("="*50)
try:
    import torch
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            name = torch.cuda.get_device_name(i)
            vram = torch.cuda.get_device_properties(i).total_memory/1e9
            print(f"GPU {i}: {name}  VRAM: {vram:.1f} GB")
        print(f"CUDA version: {torch.version.cuda}")
    else:
        print("CUDA not available")
except ImportError:
    print("PyTorch not installed")

print("\n" + "="*50)
print("RECOMMENDATION")
print("="*50)
phys = psutil.cpu_count(logical=False)
avail = psutil.virtual_memory().available/1e9
max_ram = max(1, int(avail / 3.0))
max_cpu = max(1, phys - 1)
rec = min(max_ram, max_cpu)
print(f"Available RAM  : {avail:.1f} GB")
print(f"RAM limit      : {max_ram} workers  (3 GB each)")
print(f"CPU limit      : {max_cpu} workers  (physical-1)")
print(f"\n>>> Recommended workers: {rec} <<<")

CPU
Physical cores : 14
Logical cores  : 20
Frequency      : 1506 / 2300 MHz

RAM
Total          : 34.1 GB
Available      : 20.6 GB
Used           : 13.5 GB  (39.5%)
Swap total     : 14.5 GB

DISK
C:\         Total:209GB  Free:50GB
E:\         Total:367GB  Free:127GB
F:\         Total:447GB  Free:208GB

GPU
GPU 0: NVIDIA GeForce RTX 3070 Laptop GPU  VRAM: 8.6 GB
CUDA version: 12.1

RECOMMENDATION
Available RAM  : 20.4 GB
RAM limit      : 6 workers  (3 GB each)
CPU limit      : 13 workers  (physical-1)

>>> Recommended workers: 6 <<<


In [ ]:
"""
اسکریپت بررسی مشخصات سیستم
برای تصمیم‌گیری درباره تعداد worker های موازی در preprocessing
"""

import os
import sys
import platform
import subprocess
import time

def separator(title):
    print(f"\n{'='*55}")
    print(f"  {title}")
    print('='*55)

# ============================================================
#  ۱. اطلاعات CPU
# ============================================================
separator("CPU")

# تعداد هسته‌های منطقی و فیزیکی
try:
    import psutil
    physical = psutil.cpu_count(logical=False)
    logical  = psutil.cpu_count(logical=True)
    print(f"  هسته فیزیکی (Physical cores) : {physical}")
    print(f"  هسته منطقی  (Logical  cores) : {logical}")
    print(f"  → برای multiprocessing از Physical استفاده می‌شود")
except ImportError:
    logical = os.cpu_count()
    print(f"  تعداد CPU (os.cpu_count)      : {logical}")
    print(f"  [psutil نصب نیست — هسته فیزیکی قابل تشخیص نیست]")
    physical = None

# نام CPU
try:
    if platform.system() == "Windows":
        result = subprocess.check_output(
            "wmic cpu get Name", shell=True, text=True, stderr=subprocess.DEVNULL)
        cpu_name = [l.strip() for l in result.strip().split('\n') if l.strip() and l.strip() != 'Name']
        print(f"  مدل CPU                        : {cpu_name[0] if cpu_name else 'نامشخص'}")
    elif platform.system() == "Linux":
        with open('/proc/cpuinfo') as f:
            for line in f:
                if 'model name' in line:
                    print(f"  مدل CPU                        : {line.split(':')[1].strip()}")
                    break
    elif platform.system() == "Darwin":
        result = subprocess.check_output(
            ["sysctl", "-n", "machdep.cpu.brand_string"],
            text=True, stderr=subprocess.DEVNULL).strip()
        print(f"  مدل CPU                        : {result}")
except Exception as e:
    print(f"  مدل CPU                        : خطا در خواندن ({e})")

# فرکانس CPU
try:
    import psutil
    freq = psutil.cpu_freq()
    if freq:
        print(f"  فرکانس فعلی                   : {freq.current:.0f} MHz")
        print(f"  فرکانس حداکثر                 : {freq.max:.0f} MHz")
except Exception:
    pass

# ============================================================
#  ۲. حافظه RAM
# ============================================================
separator("RAM")

try:
    import psutil
    mem = psutil.virtual_memory()
    print(f"  RAM کل                         : {mem.total / 1e9:.1f} GB")
    print(f"  RAM آزاد (available)           : {mem.available / 1e9:.1f} GB")
    print(f"  RAM در حال استفاده             : {mem.used / 1e9:.1f} GB  ({mem.percent:.1f}%)")
    print(f"  RAM آزاد (free)                : {mem.free / 1e9:.1f} GB")

    # swap
    swap = psutil.swap_memory()
    print(f"  Swap/Page file کل              : {swap.total / 1e9:.1f} GB")
    print(f"  Swap در حال استفاده            : {swap.used / 1e9:.1f} GB  ({swap.percent:.1f}%)")

    available_gb = mem.available / 1e9
except ImportError:
    print("  [psutil نصب نیست — اطلاعات دقیق RAM در دسترس نیست]")
    available_gb = None

    if platform.system() == "Linux":
        try:
            with open('/proc/meminfo') as f:
                for line in f:
                    if line.startswith(('MemTotal', 'MemAvailable', 'MemFree')):
                        print(f"  {line.strip()}")
        except Exception:
            pass

# ============================================================
#  ۳. دیسک (فضای ذخیره‌سازی)
# ============================================================
separator("دیسک — فضای ذخیره‌سازی")

try:
    import psutil
    # دیسک‌های موجود
    partitions = psutil.disk_partitions()
    for p in partitions:
        try:
            usage = psutil.disk_usage(p.mountpoint)
            print(f"  {p.mountpoint:20s}  کل: {usage.total/1e9:6.1f} GB  "
                  f"آزاد: {usage.free/1e9:6.1f} GB  "
                  f"استفاده‌شده: {usage.percent:.1f}%")
        except (PermissionError, OSError):
            pass
except ImportError:
    if platform.system() == "Windows":
        result = subprocess.check_output("wmic logicaldisk get size,freespace,caption",
                                         shell=True, text=True, stderr=subprocess.DEVNULL)
        print(result)
    else:
        result = subprocess.check_output("df -h", shell=True, text=True)
        print(result)

# ============================================================
#  ۴. سیستم‌عامل و Python
# ============================================================
separator("سیستم‌عامل و Python")

print(f"  سیستم‌عامل                      : {platform.system()} {platform.release()}")
print(f"  نسخه کامل OS                    : {platform.version()[:80]}")
print(f"  معماری                          : {platform.machine()}")
print(f"  نسخه Python                     : {sys.version}")
print(f"  مسیر Python                     : {sys.executable}")

# ============================================================
#  ۵. GPU (اختیاری — برای آموزش مدل)
# ============================================================
separator("GPU (برای آموزش مدل — preprocessing از CPU استفاده می‌کند)")

try:
    import torch
    if torch.cuda.is_available():
        print(f"  ✅ CUDA در دسترس است")
        for i in range(torch.cuda.device_count()):
            name = torch.cuda.get_device_name(i)
            mem_total = torch.cuda.get_device_properties(i).total_memory / 1e9
            print(f"  GPU {i}: {name}  |  VRAM: {mem_total:.1f} GB")
        print(f"  نسخه CUDA                       : {torch.version.cuda}")
    else:
        print(f"  ❌ CUDA در دسترس نیست (CPU-only PyTorch نصب شده یا GPU ندارد)")
except ImportError:
    print(f"  [PyTorch نصب نیست — GPU قابل بررسی نیست]")
    # fallback: nvidia-smi
    try:
        result = subprocess.check_output(
            "nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
            shell=True, text=True, stderr=subprocess.DEVNULL)
        print(f"  nvidia-smi: {result.strip()}")
    except Exception:
        print(f"  nvidia-smi در دسترس نیست")

# ============================================================
#  ۶. پکیج‌های مورد نیاز preprocessing
# ============================================================
separator("پکیج‌های نصب‌شده (مورد نیاز preprocessing)")

packages = [
    ('numpy',         'numpy'),
    ('scipy',         'scipy'),
    ('mne',           'mne'),
    ('mne_icalabel',  'mne_icalabel'),
    ('matplotlib',    'matplotlib'),
    ('pywt',          'pywt'),
    ('psutil',        'psutil'),
    ('torch',         'torch'),
    ('sklearn',       'sklearn'),
]

for display_name, import_name in packages:
    try:
        mod = __import__(import_name)
        ver = getattr(mod, '__version__', 'نامشخص')
        print(f"  ✅ {display_name:15s} : {ver}")
    except ImportError:
        print(f"  ❌ {display_name:15s} : نصب نشده")

# ============================================================
#  ۷. تست سرعت واقعی multiprocessing
# ============================================================
separator("تست سرعت multiprocessing")

import multiprocessing
import numpy as np

def _worker_test(n):
    """یک عملیات سنگین شبیه spatial mapping"""
    arr = np.random.randn(2000, 28).astype(np.float32)
    result = np.dot(arr.T, arr)
    return result.sum()

if __name__ == '__main__':
    n_logical = os.cpu_count() or 1
    test_tasks = list(range(8))

    print(f"  اجرای 8 وظیفه موازی (شبیه‌سازی spatial mapping)...")

    # Serial
    t0 = time.time()
    for t in test_tasks:
        _worker_test(t)
    serial_time = time.time() - t0
    print(f"  زمان Serial (یک هسته)          : {serial_time:.2f} s")

    # Parallel با 2 worker
    for n_w in [2, 4, min(6, n_logical)]:
        if n_w > n_logical:
            continue
        t0 = time.time()
        try:
            ctx = multiprocessing.get_context('spawn')
            with ctx.Pool(n_w) as pool:
                pool.map(_worker_test, test_tasks)
            p_time = time.time() - t0
            speedup = serial_time / p_time
            print(f"  زمان Parallel ({n_w} worker)         : {p_time:.2f} s  "
                  f"→ speedup: {speedup:.1f}x")
        except Exception as e:
            print(f"  خطا با {n_w} worker: {e}")

# ============================================================
#  ۸. خلاصه توصیه
# ============================================================
separator("خلاصه و توصیه برای تعداد Worker")

try:
    import psutil
    phys = psutil.cpu_count(logical=False) or 1
    avail_ram = psutil.virtual_memory().available / 1e9

    # هر worker حدود 3 GB RAM نیاز دارد (از تحلیل قبلی)
    ram_per_worker = 3.0
    max_by_ram  = max(1, int(avail_ram / ram_per_worker))
    max_by_cpu  = max(1, phys - 1)  # یک هسته برای OS
    recommended = min(max_by_ram, max_by_cpu)

    print(f"  RAM آزاد فعلی                  : {avail_ram:.1f} GB")
    print(f"  RAM مورد نیاز هر subject       : ~3.0 GB (peak)")
    print(f"  حداکثر worker بر اساس RAM     : {max_by_ram}")
    print(f"  حداکثر worker بر اساس CPU     : {max_by_cpu} (physical-1)")
    print(f"\n  ✅ تعداد worker توصیه‌شده      : {recommended}")
    print(f"\n  کد multiprocessing برای شما:")
    print(f"  ─────────────────────────────────────────────────")
    print(f"  from multiprocessing import Pool, get_context")
    print(f"  subjects = [f'VP{{i:03d}}' for i in range(1, 27)]")
    print(f"  ctx = get_context('spawn')  # امن‌تر از fork")
    print(f"  with ctx.Pool({recommended}) as pool:")
    print(f"      pool.map(process_subject, subjects)")
    print(f"  ─────────────────────────────────────────────────")

except ImportError:
    print("  [psutil نصب نیست — توصیه دقیق ممکن نیست]")
    print(f"  CPU count: {os.cpu_count()}")
    print(f"  توصیه پیش‌فرض: n_workers = max(1, cpu_count // 2)")

print("\n" + "="*55)
print("  اسکریپت به پایان رسید — نتایج بالا را کپی کنید")
print("="*55 + "\n")


  CPU
  هسته فیزیکی (Physical cores) : 14
  هسته منطقی  (Logical  cores) : 20
  → برای multiprocessing از Physical استفاده می‌شود
  مدل CPU                        : 12th Gen Intel(R) Core(TM) i7-12700H
  فرکانس فعلی                   : 1506 MHz
  فرکانس حداکثر                 : 2300 MHz

  RAM
  RAM کل                         : 34.1 GB
  RAM آزاد (available)           : 20.5 GB
  RAM در حال استفاده             : 13.6 GB  (39.9%)
  RAM آزاد (free)                : 20.5 GB
  Swap/Page file کل              : 14.5 GB
  Swap در حال استفاده            : 0.8 GB  (5.7%)

  دیسک — فضای ذخیره‌سازی
  C:\                   کل:  208.6 GB  آزاد:   50.3 GB  استفاده‌شده: 75.9%
  E:\                   کل:  367.0 GB  آزاد:  126.9 GB  استفاده‌شده: 65.4%
  F:\                   کل:  447.5 GB  آزاد:  207.8 GB  استفاده‌شده: 53.6%

  سیستم‌عامل و Python
  سیستم‌عامل                      : Windows 11
  نسخه کامل OS                    : 10.0.26200
  معماری                          : AMD64
  نسخه Python       